In [2]:
import os
from keras.applications.vgg16 import VGG16
from keras.utils import plot_model
from pipeline_utils import build_model, data_aug, fit_model, curves, evaluate_confusion_matrix

In [3]:
# 1. Configuración de Rutas Relativas y Parámetros
train_path = "./data/train/"
validation_path = "./data/validation/"
output_dir = "./outputs/"
os.makedirs(output_dir, exist_ok=True)

IMG_R, IMG_C = 224, 224
BS = 35 
BS_VAL = 5 
EPOCHS = 45 
NUM_TOPICS = 5
DROP_RATE = 0.3
SHIFT_FRACTION = 0.3
LR = 1e-5

In [4]:
# 2. Inicialización de Transfer Learning (VGG16)
vgg_conv = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_R, IMG_C, 3))

for layer in vgg_conv.layers[:]:
    layer.trainable = False

# 3. Construcción del Modelo
model = build_model(vgg_conv, NUM_TOPICS, DROP_RATE, LR)
# plot_model(model, to_file=f'{output_dir}/model_summary.jpg') # Comentamos esto temporalmente

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [5]:
# 4. Procesamiento de Datos (Data Augmentation)
batches, val_batches = data_aug(train_path, validation_path, BS, BS_VAL, SHIFT_FRACTION, IMG_R, IMG_C)

# 5. Entrenamiento
model, histories, score = fit_model(model, batches, val_batches, BS, BS_VAL, EPOCHS)
print(f'Accuracy: {score[1]} \nLoss: {score[0]}')

Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.


ValueError: The PyDataset has length 0

In [ ]:
# 6. Evaluación y Gráficas
curves(histories, EPOCHS, output_dir=output_dir)

CLASS_LABELS = list(val_batches.class_indices.keys())
classification_rep = evaluate_confusion_matrix(model, val_batches, CLASS_LABELS, output_dir=output_dir)
print(classification_rep)

# 7. Guardado del Modelo
model.save(f'{output_dir}/DeepFashionModelVGG16.h5')